In [ ]:
import sys
sys.path.append('..')
sys.path.append('../Stuff')

from Constants import db

cursor = db.cursor()

In [ ]:
from Stuff.DataPrep.DataPrep import DataPrep
from Stuff.DataPrep.PrepMap import standard_prep_map

dataPrep = DataPrep(standard_prep_map)

In [ ]:
from DBTypes import *

pitches = DB_PitchStatcast.Select_From_DB(
    cursor=cursor,
    conditional=dataPrep.conditional_statement + "AND LevelId=1",
    values=()
)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

run_values = [p.RunValueHitter for p in pitches]
sorted_data = np.sort(run_values)
cdf = np.arange(1, len(sorted_data) + 1) / len(sorted_data)

In [ ]:
import torch
from Buckets import *

run_buckets = torch.bucketize(torch.tensor(run_values), BUCKET_PITCHVALUE, right=True)

In [ ]:
labels = [f"(-∞, {BUCKET_PITCHVALUE[0]}]"]
for i in range(BUCKET_PITCHVALUE.size(0)-1):
    left = f"{BUCKET_PITCHVALUE[i]:.3f}"
    right = f"{BUCKET_PITCHVALUE[i+1]:.3f}"

    if left == "-0.000" and right == "0.000":
        label = "Exactly 0"
    else:
        label = f"({left}, {right}]"
    labels.append(label)
    
labels.append(f"({BUCKET_PITCHVALUE[-1].item()}, +∞]")

In [ ]:
num_buckets = BUCKET_PITCHVALUE.size(0) - 1
counts = torch.bincount(run_buckets, minlength=num_buckets).numpy()

plt.figure(figsize=(16, 7))

x_range = range(len(labels))

bars = plt.bar(x_range, counts,
               color='skyblue',
               edgecolor='navy',
               linewidth=1.2,
               alpha=0.9)

plt.title('Histogram — Manually Defined Buckets', fontsize=15, pad=15)
plt.xlabel('Value Buckets', fontsize=12)
plt.ylabel('Count', fontsize=12)

# Use your custom labels on the x-axis
plt.xticks(x_range, labels, rotation=45, ha='right', fontsize=10)

# Add count on top of each bar
max_count = counts.max()
for bar in bars:
    height = bar.get_height()
    if height > 0:
        plt.text(bar.get_x() + bar.get_width()/2,
                 height + max_count * 0.015,
                 f'{int(height)}',
                 ha='center', va='bottom',
                 fontsize=10,
                 fontweight='bold')

plt.grid(axis='y', linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()